in this file we will downsample our tiff files 

In [2]:
import os 
import rasterio
from rasterio.enums import Resampling
import numpy as np 

In [3]:
def find_all_tiff_files(root_folder):
    tiff_files = []
    for root, dirs, files in os.walk(root_folder):
        for file in files:
            if file.lower().endswith(('.tif', '.tiff')):
                tiff_files.append(os.path.join(root, file))
    return tiff_files

In [4]:
def resample_to_500m(tif_path, output_folder):
    with rasterio.open(tif_path) as src:
        src_res_x = src.transform.a
        src_res_y = abs(src.transform.e)

        print(f"Original resolution (in degrees): {src_res_x} , {src_res_y}")

        if src_res_x == 500 and src_res_y == 500:
            print(f"Already at 500m resolution: {tif_path}")
            return

        central_lat = (src.bounds.top + src.bounds.bottom) / 2
        meters_per_degree = 111320

        scale_x = src_res_x * meters_per_degree * abs(np.cos(np.radians(central_lat))) / 500
        scale_y = src_res_y * meters_per_degree / 500

        new_width = int(src.width * scale_x)
        new_height = int(src.height * scale_y)

        resample_method = Resampling.bilinear

        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=resample_method
        )

        new_transform = src.transform * src.transform.scale(
            src.width / new_width,
            src.height / new_height
        )

        profile = src.profile
        profile.update({
            'transform': new_transform,
            'height': new_height,
            'width': new_width
        })

        os.makedirs(output_folder, exist_ok=True)
        filename = os.path.basename(tif_path)
        output_path = os.path.join(output_folder, filename)

        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(data)

        print(f"Resampled and saved: {output_path}")

In [5]:
import os 
os.listdir("../Resmapled/")

['climat',
 'human_activities.tif',
 'mammals.tif',
 'organisme',
 'ParentMaterial',
 'relief',
 'saptialPosition',
 'soil_Rh_mean.tif',
 'soil_Rs_CV.tif',
 'soil_Rs_mean.tif',
 'wc2.1_30s_prec_01.tif',
 'wc2.1_30s_srad_01.tif',
 'wc2.1_30s_tmax_01.tif',
 'wc2.1_30s_tmin_01.tif',
 'wc2.1_30s_vapr_01.tif',
 'wc2.1_30s_wind_01.tif']

In [ ]:
print(find_all_tiff_files("..\Maps\\"))

['..\\Maps\\Climat\\prec\\wc2.1_30s_prec_01.tif', '..\\Maps\\Climat\\srad\\wc2.1_30s_srad_01.tif', '..\\Maps\\Climat\\tmax\\wc2.1_30s_tmax_01.tif', '..\\Maps\\Climat\\tmin\\wc2.1_30s_tmin_01.tif', '..\\Maps\\Climat\\vapr\\wc2.1_30s_vapr_01.tif', '..\\Maps\\Climat\\wind\\wc2.1_30s_wind_01.tif', '..\\Maps\\organisme\\fauna\\amphibians.tif', '..\\Maps\\organisme\\fauna\\birds.tif', '..\\Maps\\organisme\\fauna\\mammals.tif', '..\\Maps\\organisme\\fauna\\reptiles.tif', '..\\Maps\\organisme\\human\\human_activities.tif', '..\\Maps\\organisme\\Microbial\\SoilResp_HeterotrophicResp_1928\\data\\soil_Rh_mean.tif', '..\\Maps\\organisme\\Microbial\\SoilResp_HeterotrophicResp_1928\\data\\soil_Rs_CV.tif', '..\\Maps\\organisme\\Microbial\\SoilResp_HeterotrophicResp_1928\\data\\soil_Rs_mean.tif', '..\\Maps\\organisme\\Microbial\\SoilResp_HeterotrophicResp_1928\\data\\soil_Rs_SD.tif', '..\\Maps\\organisme\\Vigitation\\barren.tif', '..\\Maps\\organisme\\Vigitation\\cultivated.tif', '..\\Maps\\organisme\

In [7]:
all_tiffs = find_all_tiff_files("..\Maps\\")

for tif in all_tiffs :
    same = os.path.basename(tif)

    if same in os.listdir("../Resmapled/"):
        continue
    else : 
        try : 
            print(tif)
            print("\n\n\n")
            resample_to_500m(tif, "../Resmapled/")
        except Exception as e :
            print(e)
            print("\n\n\n")

..\Maps\organisme\fauna\amphibians.tif




Original resolution (in degrees): 30000.0 , 30000.0
Unable to allocate 7.54 EiB for an array with shape (1, 3954086400, 2197123175) and data type uint8




..\Maps\organisme\fauna\birds.tif




Original resolution (in degrees): 30000.0 , 30000.0
array is too big; `arr.size * arr.dtype.itemsize` is larger than the maximum possible size.




..\Maps\organisme\fauna\reptiles.tif




Original resolution (in degrees): 30000.0 , 30000.0
Unable to allocate 7.54 EiB for an array with shape (1, 3954086400, 2197123175) and data type uint8




..\Maps\organisme\Microbial\SoilResp_HeterotrophicResp_1928\data\soil_Rs_SD.tif




Original resolution (in degrees): 0.008333333300000002 , 0.008333333300000002
Resampled and saved: ../Resmapled/soil_Rs_SD.tif
..\Maps\organisme\Vigitation\barren.tif




Original resolution (in degrees): 0.008333333333333335 , 0.008333333333333335
Resampled and saved: ../Resmapled/barren.tif
..\Maps\organisme\Vigitation\cultivated.